# Qwen captions and text-alignment embeddings

This report verifies the text artifacts used by the Text Alignment pipeline. Qwen generates captions from stimulus images; it does not produce the embedding used by `train_eeg_align_text.yaml`. The current alignment target is a mean-pooled `t5_base` embedding.

In [1]:
from collections import Counter
import json
from pathlib import Path
import torch

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
CAPTION_PATH = PROJECT_ROOT / 'data/things-eeg2/captions/local.jsonl'
rows = [json.loads(line) for line in CAPTION_PATH.open()]
assert len({row['path'] for row in rows}) == len(rows)
assert all(row['caption'].strip() for row in rows)
Counter(row['split'] for row in rows)

Counter({'train': 16540, 'test': 200})

## Artifact roles

| Artifact | Producer | Default location | Used by alignment? |
|---|---|---|---|
| Caption | Qwen2.5-VL-7B-Instruct | `data/things-eeg2/captions/local.jsonl` | Indirectly |
| Text embedding | T5 / CLIP / Gemma encoder | `tensorcache/<encoder>/<split>/...` | Yes, selected encoder only |
| Statistics | `generate_stats.py --config-name=generate_text_stats` | `statistics/datasets/things-eeg2/<split>/` | Yes |

The effective `train_eeg_align_text` configuration selects `t5_base`, with alignment enabled and reconstruction disabled.

In [2]:
models = ['t5_base', 'clip_vitl14_text', 'gemma_embedding_300m']
for model in models:
    for split in ['train', 'test']:
        stat_path = PROJECT_ROOT / 'statistics/datasets/things-eeg2' / split / f'{model}.pt'
        stats = torch.load(stat_path, map_location='cpu', weights_only=True)
        print(model, split, tuple(stats['mean'].shape), tuple(stats['std'].shape))

t5_base train (768,) (768,)
t5_base test (768,) (768,)
clip_vitl14_text train (768,) (768,)
clip_vitl14_text test (768,) (768,)
gemma_embedding_300m train (768,) (768,)
gemma_embedding_300m test (768,) (768,)


## Verified current state

The current caption file contains 16,740 unique, non-empty records: 16,540 training images and 200 test images. All referenced images exist. Each of the three downstream encoders has 16,540 training and 200 test cached embeddings, and each has matching 768-dimensional float32 statistics.

The JSONL file does not record the Qwen model or prompt per record, so its generation provenance cannot be proven from the artifact alone. The model and prompt documented here are the current defaults in `generate_text_captions_local.yaml`; regeneration is only needed if stricter provenance is required.